# EDA for `chunks_labeled.json` files

This notebook loads every `chunks_labeled.json` file under the DeepSeek math rollouts directory into a single pandas dataframe. Each dataframe row corresponds to one dictionary from a JSON file, with extra columns for the source metadata inferred from the parent folders.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

In [ ]:
# Notebook location: thought-anchors/EDA/chunks_labeled_eda.ipynb
# Project root:       thought-anchors/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "EDA" else Path.cwd()

BASE_DIR = (
    PROJECT_ROOT
    / "math_rollouts"
    / "deepseek-r1-distill-qwen-14b"
    / "temperature_0.6_top_p_0.95"
)

BASE_DIR

In [ ]:
def load_chunks_labeled(base_dir: Path) -> pd.DataFrame:
    """Load all chunks_labeled.json files below base_dir into one dataframe."""
    rows = []
    source_files = sorted(base_dir.rglob("chunks_labeled.json"))

    for source_file in source_files:
        relative_parts = source_file.relative_to(base_dir).parts
        solution_type = relative_parts[0] if len(relative_parts) >= 3 else None
        problem = relative_parts[1] if len(relative_parts) >= 3 else None
        problem_id = problem[len("problem_"):] if isinstance(problem, str) and problem.startswith("problem_") else problem

        with source_file.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            data = [data]

        for row_in_file, item in enumerate(data):
            if not isinstance(item, dict):
                item = {"value": item}

            rows.append(
                {
                    **item,
                    "source_file": str(source_file),
                    "source_relative_path": str(source_file.relative_to(base_dir)),
                    "source_dir": str(source_file.parent),
                    "solution_type": solution_type,
                    "problem": problem,
                    "problem_id": problem_id,
                    "row_in_file": row_in_file,
                    "model": "deepseek-r1-distill-qwen-14b",
                    "sampling": "temperature_0.6_top_p_0.95",
                }
            )

    return pd.DataFrame(rows)


df = load_chunks_labeled(BASE_DIR)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## Source Coverage

In [ ]:
coverage = (
    df.groupby("solution_type")
    .agg(
        rows=("chunk_idx", "size"),
        problems=("problem", "nunique"),
        files=("source_file", "nunique"),
    )
    .sort_values("rows", ascending=False)
)

coverage

In [ ]:
problem_coverage = (
    df.groupby(["solution_type", "problem"])
    .size()
    .rename("chunks")
    .reset_index()
    .sort_values(["solution_type", "chunks"], ascending=[True, False])
)

problem_coverage.head(20)

## Label and Tag Distributions

In [ ]:
if "function_tags" in df.columns:
    tags = df.explode("function_tags")
    display(tags["function_tags"].value_counts().to_frame("count"))
    display(pd.crosstab(tags["solution_type"], tags["function_tags"]))

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
df[numeric_cols].describe().T

In [ ]:
importance_cols = [
    col for col in df.columns
    if "importance" in col or col in {"accuracy", "different_trajectories_fraction", "overdeterminedness"}
]

if importance_cols:
    df.groupby("solution_type")[importance_cols].mean(numeric_only=True).T

## Importance by Function Tag and Chunk Position

In [ ]:
# Filtering controls for the plotting dataframe.
# Use None or [] for all solution types, a string for one solution type,
# or a list/tuple/set for multiple solution types.
SOLUTION_TYPE_FILTER = None
# SOLUTION_TYPE_FILTER = "correct_base_solution"
# SOLUTION_TYPE_FILTER = ["correct_base_solution", "incorrect_base_solution"]

# Restrict to a subset if you want fewer figures.
IMPORTANCE_COLS_TO_PLOT = importance_cols
# IMPORTANCE_COLS_TO_PLOT = ["resampling_importance_accuracy", "counterfactual_importance_accuracy", "forced_importance_accuracy"]

# Used only for scaled chunk-position plots. Set to None to plot every observed scaled position.
N_POSITION_BINS = 20

In [ ]:
def _normalize_filter_values(values):
    if values is None or values == []:
        return None
    if isinstance(values, str):
        return [values]
    return list(values)


def make_importance_plot_df(
    data: pd.DataFrame,
    solution_type_filter=None,
    importance_columns=None,
) -> pd.DataFrame:
    importance_columns = importance_columns or importance_cols
    filter_values = _normalize_filter_values(solution_type_filter)

    plot_df = data.copy()
    if filter_values is not None:
        plot_df = plot_df[plot_df["solution_type"].isin(filter_values)].copy()

    if "function_tags" not in plot_df.columns:
        raise ValueError("Expected a function_tags column in df.")

    plot_df = plot_df.explode("function_tags")
    plot_df = plot_df.rename(columns={"function_tags": "function_tag"})
    plot_df["function_tag"] = plot_df["function_tag"].fillna("<missing>")

    available_importance_cols = [col for col in importance_columns if col in plot_df.columns]
    if not available_importance_cols:
        raise ValueError("No requested importance columns are present in df.")

    plot_df["chunk_idx"] = pd.to_numeric(plot_df["chunk_idx"], errors="coerce")
    for col in available_importance_cols:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

    group_keys = ["solution_type", "problem"]
    min_idx = plot_df.groupby(group_keys)["chunk_idx"].transform("min")
    max_idx = plot_df.groupby(group_keys)["chunk_idx"].transform("max")
    span = max_idx - min_idx
    plot_df["chunk_idx_scaled"] = ((plot_df["chunk_idx"] - min_idx) / span.where(span != 0, 1)).fillna(0)

    return plot_df.dropna(subset=["chunk_idx", "chunk_idx_scaled"])


plot_df = make_importance_plot_df(df, SOLUTION_TYPE_FILTER, IMPORTANCE_COLS_TO_PLOT)
plot_df[["solution_type", "problem", "chunk_idx", "chunk_idx_scaled", "function_tag"]].head()

In [ ]:
def plot_importance_by_function_tag(
    plot_df: pd.DataFrame,
    x_col: str,
    importance_columns=None,
    title_prefix: str = "",
    max_tags: int = 12,
):
    importance_columns = [col for col in (importance_columns or importance_cols) if col in plot_df.columns]
    top_tags = plot_df["function_tag"].value_counts().head(max_tags).index
    df_to_plot = plot_df[plot_df["function_tag"].isin(top_tags)].copy()

    for col in importance_columns:
        summarized = (
            df_to_plot.groupby([x_col, "function_tag"], as_index=False)[col]
            .mean()
            .sort_values(x_col)
        )

        fig, ax = plt.subplots(figsize=(11, 6))
        for tag, tag_df in summarized.groupby("function_tag"):
            ax.plot(tag_df[x_col], tag_df[col], marker="o", linewidth=1.5, markersize=3, label=tag)

        ax.set_title(f"{title_prefix}{col} by function_tag")
        ax.set_xlabel(x_col)
        ax.set_ylabel(f"mean {col}")
        ax.grid(True, alpha=0.25)
        ax.legend(title="function_tag", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

### Raw `chunk_idx`

In [ ]:
plot_importance_by_function_tag(
    plot_df=plot_df,
    x_col="chunk_idx",
    importance_columns=IMPORTANCE_COLS_TO_PLOT,
    title_prefix="Raw chunk_idx: ",
)

### Scaled `chunk_idx` Within Each `solution_type` - `problem`

In [ ]:
scaled_plot_df = plot_df.copy()

if N_POSITION_BINS is not None:
    scaled_plot_df["chunk_position_bin"] = pd.cut(
        scaled_plot_df["chunk_idx_scaled"],
        bins=N_POSITION_BINS,
        include_lowest=True,
    )
    scaled_plot_df["chunk_position"] = scaled_plot_df["chunk_position_bin"].apply(lambda interval: interval.mid)
else:
    scaled_plot_df["chunk_position"] = scaled_plot_df["chunk_idx_scaled"]

plot_importance_by_function_tag(
    plot_df=scaled_plot_df,
    x_col="chunk_position",
    importance_columns=IMPORTANCE_COLS_TO_PLOT,
    title_prefix="Scaled chunk_idx: ",
)

## Optional Export

In [ ]:
# Uncomment if you want a reusable flat file for downstream analysis.
# output_path = Path.cwd() / "chunks_labeled_combined.parquet"
# df.to_parquet(output_path, index=False)
# output_path